# Gate v4-mixed-r16 at lambda=0.75 and clear the production check

Three steps in one GPU session, in this order:

1. **Fetch VIVOS** -- tier 2 OOD needs it, and the gate cannot run without it.
2. **Gate at lambda=0.75** -- produces a run dir `merge_and_push.py` accepts
   as-is (gated `gate_results.json`, adapter baked at 0.75, `config.json` and
   `lambda_sweep.csv` copied alongside).
3. **Score v3-r16 at lambda=0.5 on the synthetic 426** -- the model production
   actually runs. `check_no_regression_vs_production` needs its predictions,
   and the stored `Outputs/v3-r16/audit/*.csv` were scored at **lambda=1.0**,
   not 0.5 (that run gated under the old "largest within budget" rule; 0.5 was
   a later human pick). Comparing against those means comparing against a
   model nobody serves.

lambda=0.75 is a **human override** of `select_lambda`, which still picks 0.5.
The rule optimises val CER against OOD CER, and val CER cannot see
`english_token_retention` -- the axis production regressed on, where 0.75 adds
~3.2pp over 0.5 on every slice. See SESSIONS.md for the full curve.

Nothing is pushed here. Publishing is a separate, deliberate step.

## 1. Repo

In [ ]:
import os

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Attach: **`lambda-sweep-artifacts`** (re-upload the current zip -- it now also
carries `metrics/baseline.json`, `metrics/lambda_sweep.csv` and
`audit/predictions_baseline_real.csv`, all three of which the gate reads),
**`youtube-meetings`**, **`paid-dataset-v2`**, **`real-meetings-bench`**, and
**`h6-artifacts`** (holds `v3-r16/` with its raw `checkpoints/best`, for step 3).

Mount nesting differs per dataset -- read the paths off this cell, never from a
previous notebook.

In [ ]:
!ls -la /kaggle/input/datasets/*/*

In [ ]:
ARTIFACTS   = "/kaggle/input/datasets/<user>/lambda-sweep-artifacts/lambda-sweep-artifacts"  # edit
YOUTUBE_SRC = "/kaggle/input/datasets/<user>/youtube-meetings/youtube-meetings"              # edit
PAID_SRC    = "/kaggle/input/datasets/<user>/paid-dataset-v2/paid-dataset-v2"                # edit
REAL_BENCH  = "/kaggle/input/datasets/<user>/real-meetings-bench/real-meetings-bench"        # edit
H6          = "/kaggle/input/datasets/<user>/h6-artifacts/h6-artifacts"                      # edit

RUN_DIR    = f"{ARTIFACTS}/v4-mixed-r16"
V3_RUN_DIR = f"{H6}/v3-r16"

for label, p in [("v4 config.json", f"{RUN_DIR}/config.json"),
                 ("v4 checkpoints/best", f"{RUN_DIR}/checkpoints/best"),
                 ("v4 metrics/baseline.json", f"{RUN_DIR}/metrics/baseline.json"),
                 ("v4 metrics/lambda_sweep.csv", f"{RUN_DIR}/metrics/lambda_sweep.csv"),
                 ("v4 audit/predictions_baseline_real.csv",
                  f"{RUN_DIR}/audit/predictions_baseline_real.csv"),
                 ("v4 validated_manifest.jsonl", f"{RUN_DIR}/validated_manifest.jsonl"),
                 ("v3 config.json", f"{V3_RUN_DIR}/config.json"),
                 ("v3 checkpoints/best", f"{V3_RUN_DIR}/checkpoints/best"),
                 ("youtube", YOUTUBE_SRC), ("paid", PAID_SRC), ("real-bench", REAL_BENCH)]:
    print(f"{os.path.exists(p)}\t{label}")

## 3. Rebuild `mixed-noisy-v1`

`data.dataset_path` was `/kaggle/working/...` and is not persisted, so the
merged audio tree is reassembled by the same script the original run used --
it re-runs every gate (disjoint `meeting_id`, disjoint audio dirs, `verified:
true` everywhere) before copying a byte.

`--out` must not already exist non-empty. Fresh session: fine. Re-running this
cell in the same session: not.

In [ ]:
MIXED = "/kaggle/working/dataset/mixed-noisy-v1"
!python -m scripts.build_mixed_dataset --sources {PAID_SRC} {YOUTUBE_SRC} --out {MIXED}

AUDIO_ROOT = f"{MIXED}/audio"
print("AUDIO_ROOT exists:", os.path.exists(AUDIO_ROOT))

## 4. Fetch VIVOS (tier 2 OOD)

`--smoke` first: `fetch_vivos.py` has two routes (parquet, then a raw-tarball
fallback) and its own docstring flags both as historically fragile. Cheap to
check before spending the full download.

In [ ]:
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"
VIVOS = "/kaggle/working/dataset/vivos"

!python -m scripts.fetch_vivos --out {VIVOS} --split test --smoke

In [ ]:
!python -m scripts.fetch_vivos --out {VIVOS} --split test
print("VIVOS exists:", os.path.exists(f"{VIVOS}/audio"))

## 5. Gate at lambda=0.75

Bakes the adapter at 0.75, then runs tiers 1 / 2 / 4a. Watch two lines:

- `tier2_ood measured ... sweep row says ... delta` -- must be below 1e-9.
  `merge_and_push.check_provenance` cross-checks these, and greedy decode over
  the same OOD split should reproduce the sweep exactly. A real gap is a
  finding to chase, not something to edit away.
- `overall_pass=True`.

In [ ]:
GATED = "/kaggle/working/v4-mixed-r16-lambda0.75"

!python -m scripts.gate_at_lambda \
    --run-dir {RUN_DIR} \
    --lam 0.75 \
    --audio-root {AUDIO_ROOT} \
    --ood-eval-path {VIVOS} \
    --real-bench-path {REAL_BENCH} \
    --out-dir {GATED}

## 6. Score v3-r16 at lambda=0.5 -- the model production runs

Without this, `check_no_regression_vs_production` compares the candidate
against v3-r16 at lambda=1.0, which is stronger on synthetic than the
published lambda=0.5 and is not what any server serves.

Scores the same 654-segment test split, so the resulting CSV joins with the
candidate's on `(meeting_id, segment_id)` across **both** sources. The youtube
228 are already known at this lambda (CER 0.1205) -- if that number does not
reappear here, stop: something differs between the two runs.

In [ ]:
V3_L05 = "/kaggle/working/v3-r16-lambda0.5"

!python -m scripts.eval_v4_mixed_at_lambda \
    --run-dir {V3_RUN_DIR} \
    --audio-root {AUDIO_ROOT} \
    --lam 0.5 \
    --out-dir {V3_L05}

## 7. Run the production check on its own

`merge_and_push.py` runs this before merging anyway, but running it here means
a failure costs seconds instead of an fp32 CPU merge. It raises on a
regression; no output means the candidate is clear.

In [ ]:
from pathlib import Path
from scripts.merge_and_push import check_no_regression_vs_production

check_no_regression_vs_production(
    Path(f"{GATED}/audit/predictions_tier1_in_domain.csv"),
    Path(f"{V3_L05}/lambda0.5/predictions_tier1_in_domain.csv"),
)

## 8. What to save

Download `{GATED}` and `{V3_L05}` before the session ends -- the gated run dir
is what `merge_and_push.py` consumes, and the v3 predictions are the
production reference every future candidate has to clear.

Publishing is **not** part of this notebook. Do it deliberately, with
`HF_TOKEN` set, once the two lines in step 5 read the way they should.

In [ ]:
!ls -R {GATED} | head -40
!cat {GATED}/metrics/gate_results.json